# Chain-wise WAIC, DIC, and AIC proxy

This notebook compares the three paper models using all 10 chains. Each saved chain contains 5,000 draws (already thinned by 2 during fitting). Selecting `[:, ::15]` leaves exactly 334 draws per chain and gives an effective thinning interval of 30 original MCMC iterations.

Reported uncertainty is the sample standard deviation across the 10 independently fitted chains. The paper uses WAIC and DIC. Classical AIC requires an MLE, which these posterior files do not contain, so `AIC_proxy_at_posterior_mean` is reported only as a diagnostic and must not be described as formal AIC.


In [13]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import pyreadr
from scipy.special import expit, logsumexp
from joblib import Parallel, delayed, parallel_config
from tqdm.auto import tqdm

BASE_DIR = Path(r"D:/77/Research/temp/snow")
OUTPUT_CSV = BASE_DIR / "model_comparison_chain_sd.csv"
CHECKPOINT_JSON = BASE_DIR / "model_comparison_chain_checkpoint.json"
N_CHAINS = 10
N_JOBS = min(N_CHAINS, os.cpu_count() or 1)
SAVED_DRAW_STEP = 15
EXPECTED_DRAWS = 334
PERIOD = 52
EPS = 1e-12

assert BASE_DIR.exists(), BASE_DIR
print('Results directory:', BASE_DIR)
print('Parallel chain workers:', N_JOBS)


Results directory: D:\77\Research\temp\snow
Parallel chain workers: 10


In [14]:
# Load and align the data exactly as in the fitting scripts.
snow = list(pyreadr.read_r(str(BASE_DIR / 'snow_cleaned_full.Rda')).values())[0].reset_index(drop=True)
coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy(dtype=np.int8)
S, TT = y.shape
T = TT - 1
assert (S, TT) == (1618, 2704)

t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)
cos_all = np.cos(2 * np.pi * np.arange(1, TT) / PERIOD)
sin_all = np.sin(2 * np.pi * np.arange(1, TT) / PERIOD)
trend_all = t_scaled[:-1]
y_prev = y[:, :-1]
y_next = y[:, 1:]

# Five covariates in the final BYM+ longitude model.
lon_raw = coords[:, 0]
mask_na = lon_raw < -30
mask_ea = ~mask_na
lon_na = np.zeros(S)
lon_ea = np.zeros(S)
lon_na[mask_na] = (lon_raw[mask_na] - lon_raw[mask_na].mean()) / lon_raw[mask_na].std(ddof=0)
lon_ea[mask_ea] = (lon_raw[mask_ea] - lon_raw[mask_ea].mean()) / lon_raw[mask_ea].std(ddof=0)
lat = (coords[:, 1] - coords[:, 1].mean()) / coords[:, 1].std(ddof=0)

no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1
elev_raw = pd.read_csv(BASE_DIR / 'curr_elev.csv').iloc[:, 3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR / 'nnbs_elev.csv', sep='\t').iloc[:, 2].to_numpy()
keep = np.ones(S, dtype=bool)
keep[no_nbs] = False
elev_all = np.empty(S)
elev_all[keep] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std(ddof=0)

temp_df = list(pyreadr.read_r(str(BASE_DIR / 'snow_temp_full.Rda')).values())[0].reset_index(drop=True)
temp = temp_df.iloc[:, 2:].to_numpy()
temp_scaled = (temp - temp.mean()) / temp.std(ddof=0)
assert temp_scaled.shape == (S, TT)
print(f'S={S}, TT={TT}, one-step observations={S*T:,}')


S=1618, TT=2704, one-step observations=4,373,454


In [15]:
def thin334(a):
    out = np.asarray(a)[:, ::SAVED_DRAW_STEP]
    assert out.shape[1] == EXPECTED_DRAWS, out.shape
    return out.astype(np.float64, copy=False)

def load_npz_arrays(path):
    with np.load(path) as d:
        return {k: thin334(d[k]) for k in d.files}

def log_probability(phi01, phi10, t):
    # phi arrays are S x M. logaddexp avoids overflow in the logistic tails.
    log_p01 = -np.logaddexp(0.0, -phi01)
    log_q01 = -np.logaddexp(0.0, phi01)
    log_p10 = -np.logaddexp(0.0, -phi10)
    log_q10 = -np.logaddexp(0.0, phi10)
    prev0 = y_prev[:, t, None] == 0
    next1 = y_next[:, t, None] == 1
    return np.where(prev0, np.where(next1, log_p01, log_q01), np.where(next1, log_q10, log_p10))

def metrics_from_phi_builder(builder, posterior_mean_builder, parameter_count, label, n_draws):
    # WAIC is accumulated pointwise. DIC uses total deviance for each draw.
    M = n_draws
    lppd = 0.0
    p_waic = 0.0
    deviance = np.zeros(M)
    for t in tqdm(range(T), desc=label, leave=False):
        phi01, phi10 = builder(t)
        ll = log_probability(phi01, phi10, t)
        lppd += np.sum(logsumexp(ll, axis=1) - np.log(M))
        # ddof=0 matches the WAIC implementation used for the paper.
        p_waic += np.sum(np.var(ll, axis=1, ddof=0))
        deviance += -2.0 * ll.sum(axis=0)

    D_bar = deviance.mean()
    D_hat = 0.0
    for t in range(T):
        phi01, phi10 = posterior_mean_builder(t)
        D_hat += -2.0 * log_probability(phi01[:, None], phi10[:, None], t).sum()

    waic = -2.0 * (lppd - p_waic)
    dic = 2.0 * D_bar - D_hat
    aic_proxy = D_hat + 2.0 * parameter_count
    return dict(WAIC=waic, lppd=lppd, p_WAIC=p_waic, DIC=dic, D_bar=D_bar, D_hat=D_hat,
                AIC_proxy_at_posterior_mean=aic_proxy, draws=M)

def evaluate_ind(chains):
    chains = [chains] if isinstance(chains, (int, np.integer)) else list(chains)
    a01 = np.concatenate([load_npz_arrays(BASE_DIR / f'p01_ind_all_chain{c}.npz')['all_theta'] for c in chains], axis=1)
    a10 = np.concatenate([load_npz_arrays(BASE_DIR / f'p10_ind_all_chain{c}.npz')['all_theta'] for c in chains], axis=1)
    M = a01.shape[1]
    a01 = a01.reshape(4, S, M)
    a10 = a10.reshape(4, S, M)
    def build(t):
        x = (1.0, cos_all[t], sin_all[t], trend_all[t])
        return sum(x[k] * a01[k] for k in range(4)), sum(x[k] * a10[k] for k in range(4))
    m01, m10 = a01.mean(2), a10.mean(2)
    def build_mean(t):
        x = (1.0, cos_all[t], sin_all[t], trend_all[t])
        return sum(x[k] * m01[k] for k in range(4)), sum(x[k] * m10[k] for k in range(4))
    label = 'IND pooled' if len(chains) > 1 else f'IND chain {chains[0]}'
    return metrics_from_phi_builder(build, build_mean, 8*S, label, M)

def evaluate_weekly(chains, with_lon=False):
    chains = [chains] if isinstance(chains, (int, np.integer)) else list(chains)
    prefix = 'weekly_cov+lon' if with_lon else 'weekly'
    d01s = [load_npz_arrays(BASE_DIR / f'p01_{prefix}_chain{c}.npz') for c in chains]
    d10s = [load_npz_arrays(BASE_DIR / f'p10_{prefix}_chain{c}.npz') for c in chains]
    d01 = {k: np.concatenate([d[k] for d in d01s], axis=1) for k in d01s[0]}
    d10 = {k: np.concatenate([d[k] for d in d10s], axis=1) for k in d10s[0]}
    M = d01['eta'].shape[1]
    K = 8
    eta01, eta10 = d01['eta'], d10['eta']
    gamma01 = eta01[K*S:] if with_lon else None
    gamma10 = eta10[K*S:] if with_lon else None
    eta01 = eta01[:K*S].reshape(K, S, M)
    eta10 = eta10[:K*S].reshape(K, S, M)
    tau01 = d01['tau'].reshape(K, PERIOD, M)
    tau10 = d10['tau'].reshape(K, PERIOD, M)
    def build(t):
        week = t % PERIOD
        x = np.array([1,1,cos_all[t],cos_all[t],sin_all[t],sin_all[t],trend_all[t],trend_all[t]])
        p01 = sum(x[k] * eta01[k] * tau01[k, week] for k in range(K))
        p10 = sum(x[k] * eta10[k] * tau10[k, week] for k in range(K))
        if with_lon:
            z = (lon_na, lon_ea, lat, elev, temp_scaled[:, t])
            p01 += trend_all[t] * sum(z[k][:, None] * gamma01[k] for k in range(5))
            p10 += trend_all[t] * sum(z[k][:, None] * gamma10[k] for k in range(5))
        return p01, p10
    e01m, e10m = eta01.mean(2), eta10.mean(2)
    t01m, t10m = tau01.mean(2), tau10.mean(2)
    g01m = gamma01.mean(1) if with_lon else None
    g10m = gamma10.mean(1) if with_lon else None
    def build_mean(t):
        week = t % PERIOD
        x = np.array([1,1,cos_all[t],cos_all[t],sin_all[t],sin_all[t],trend_all[t],trend_all[t]])
        p01 = sum(x[k] * e01m[k] * t01m[k, week] for k in range(K))
        p10 = sum(x[k] * e10m[k] * t10m[k, week] for k in range(K))
        if with_lon:
            z = (lon_na, lon_ea, lat, elev, temp_scaled[:, t])
            p01 += trend_all[t] * sum(z[k] * g01m[k] for k in range(5))
            p10 += trend_all[t] * sum(z[k] * g10m[k] for k in range(5))
        return p01, p10
    npar = 2 * (K*S + K*PERIOD + (5 if with_lon else 0))
    model_label = 'BYM+lon' if with_lon else 'BYM'
    label = f'{model_label} pooled' if len(chains) > 1 else f'{model_label} chain {chains[0]}'
    return metrics_from_phi_builder(build, build_mean, npar, label, M)


In [16]:
# Long-running cell. Chains run in parallel; the parent process writes checkpoints. Safe to rerun.
if CHECKPOINT_JSON.exists():
    records = json.loads(CHECKPOINT_JSON.read_text(encoding='utf-8'))
    print(f'Resuming from {len(records)} completed model-chain rows')
else:
    records = []

done = {(r['model'], r['chain']) for r in records}
def run_model_chain(model, chain):
    # evaluate_* calls load_npz_arrays(), whose thin334() selects [:, ::15]
    # before the likelihood calculation. Every worker therefore uses 334 draws.
    if model == 'IND':
        metrics = evaluate_ind(chain)
    elif model == 'BYM':
        metrics = evaluate_weekly(chain, with_lon=False)
    elif model == 'BYM+lon':
        metrics = evaluate_weekly(chain, with_lon=True)
    else:
        raise ValueError(model)
    return {'model': model, 'chain': chain, **{k: float(v) for k, v in metrics.items()}}

for model in ['IND', 'BYM', 'BYM+lon']:
    pending = [chain for chain in range(N_CHAINS) if (model, chain) not in done]
    if not pending:
        print(f'{model}: all chains already completed')
        continue
    print(f'{model}: running {len(pending)} chains with {min(N_JOBS, len(pending))} workers')
    # generator_unordered returns each chain as soon as it finishes, allowing
    # the checkpoint to be updated without waiting for the slowest chain.
    with parallel_config(backend='loky', n_jobs=min(N_JOBS, len(pending)), inner_max_num_threads=1):
        completed = Parallel(return_as='generator_unordered')(
            delayed(run_model_chain)(model, chain) for chain in pending
        )
        for row in completed:
            records.append(row)
            done.add((row['model'], row['chain']))
            records.sort(key=lambda r: (r['model'], r['chain']))
            CHECKPOINT_JSON.write_text(json.dumps(records, indent=2), encoding='utf-8')
            pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False)
            print('completed:', row)

chain_results = pd.DataFrame(records).sort_values(['model', 'chain']).reset_index(drop=True)
chain_results


Resuming from 30 completed model-chain rows
IND: all chains already completed
BYM: all chains already completed
BYM+lon: all chains already completed


,model,chain,WAIC,lppd,p_WAIC,DIC,D_bar,D_hat,AIC_proxy_at_posterior_mean,draws
0,BYM,0,1.239108e+06,-606012.570274,13541.191192,1.237499e+06,1.225223e+06,1.212947e+06,1.266387e+06,334.0
1,BYM,1,1.239090e+06,-605975.062040,13570.073178,1.237222e+06,1.225173e+06,1.213125e+06,1.266565e+06,334.0
2,BYM,2,1.239126e+06,-605999.724860,13563.231396,1.237448e+06,1.225213e+06,1.212978e+06,1.266418e+06,334.0
3,BYM,3,1.239105e+06,-605996.828435,13555.630839,1.237388e+06,1.225206e+06,1.213023e+06,1.266463e+06,334.0
4,BYM,4,1.239091e+06,-605972.825900,13572.865502,1.237543e+06,1.225175e+06,1.212808e+06,1.266248e+06,334.0
5,BYM,5,1.239122e+06,-605992.736982,13568.161330,1.237558e+06,1.225209e+06,1.212861e+06,1.266301e+06,334.0
6,BYM,6,1.239099e+06,-605969.398493,13580.318117,1.237510e+06,1.225175e+06,1.212840e+06,1.266280e+06,334.0
7,BYM,7,1.239127e+06,-605989.823011,13573.519831,1.237297e+06,1.225212e+06,1.213127e+06,1.266567e+06,334.0
8,BYM,8,1.239185e+06,-605977.642895,13614.958420,1.236990e+06,1.225225e+06,1.213459e+06,1.266899e+06,334.0
9,BYM,9,1.239091e+06,-605993.953483,13551.456828,1.237180e+06,1.225198e+06,1.213216e+06,1.266656e+06,334.0


In [17]:
# Mean and sample SD across the 10 independent chains.
metrics = ['WAIC', 'DIC', 'AIC_proxy_at_posterior_mean', 'lppd', 'p_WAIC', 'D_bar', 'D_hat']
summary = chain_results.groupby('model')[metrics].agg(['mean', 'std'])
summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
summary = summary.reset_index()
summary.to_csv(BASE_DIR / 'model_comparison_summary.csv', index=False)
summary


,model,WAIC_mean,WAIC_std,DIC_mean,DIC_std,AIC_proxy_at_posterior_mean_mean,AIC_proxy_at_posterior_mean_std,lppd_mean,lppd_std,p_WAIC_mean,p_WAIC_std,D_bar_mean,D_bar_std,D_hat_mean,D_hat_std
0,BYM,1.239114e+06,28.636435,1.237364e+06,187.054413,1.266478e+06,199.978952,-605988.056637,13.869295,13569.140663,19.917631,1.225201e+06,19.717417,1.213038e+06,199.978952
1,BYM+lon,1.237804e+06,27.966251,1.236063e+06,77.193807,1.265695e+06,76.346004,-605576.360791,13.251897,13325.727496,15.173330,1.224149e+06,21.268756,1.212235e+06,76.346004
2,IND,1.277949e+06,24.217849,1.276883e+06,21.679448,1.277627e+06,6.636317,-625791.640674,4.767207,13182.927286,12.642459,1.264311e+06,11.708204,1.251739e+06,6.636317


In [18]:
# Z-tests for improvement between models, based on the 10 chain-level estimates.
# All three criteria are lower-is-better. Positive improvement means the new model is smaller/better.
from scipy.stats import norm

comparisons = [
    ('IND', 'BYM'),
    ('BYM', 'BYM+lon'),
    ('IND', 'BYM+lon'),
]
criteria = ['WAIC', 'DIC', 'AIC_proxy_at_posterior_mean']
z_rows = []

for old_model, new_model in comparisons:
    for criterion in criteria:
        old = chain_results.loc[chain_results.model == old_model, criterion].to_numpy()
        new = chain_results.loc[chain_results.model == new_model, criterion].to_numpy()
        assert len(old) == N_CHAINS and len(new) == N_CHAINS

        old_mean, new_mean = old.mean(), new.mean()
        old_sd, new_sd = old.std(ddof=1), new.std(ddof=1)
        improvement = old_mean - new_mean
        se_improvement = np.sqrt(old_sd**2 / len(old) + new_sd**2 / len(new))
        z = improvement / se_improvement if se_improvement > 0 else np.inf * np.sign(improvement)
        p_one_sided = norm.sf(z)       # H1: new model has a smaller criterion
        p_two_sided = 2 * norm.sf(abs(z))

        z_rows.append({
            'comparison': f'{old_model} -> {new_model}',
            'criterion': criterion,
            'old_mean': old_mean,
            'old_chain_sd': old_sd,
            'new_mean': new_mean,
            'new_chain_sd': new_sd,
            'improvement_old_minus_new': improvement,
            'improvement_percent': 100 * improvement / abs(old_mean),
            'se_of_improvement': se_improvement,
            'z_score': z,
            'p_one_sided_improvement': p_one_sided,
            'p_two_sided': p_two_sided,
            'significant_improvement_5pct_one_sided': bool(z > norm.ppf(0.95)),
            'significant_difference_5pct_two_sided': bool(abs(z) > norm.ppf(0.975)),
        })

z_results = pd.DataFrame(z_rows)
z_results.to_csv(BASE_DIR / 'model_comparison_z_scores.csv', index=False)
z_results[[
    'comparison', 'criterion', 'improvement_old_minus_new', 'improvement_percent',
    'z_score', 'p_one_sided_improvement', 'significant_improvement_5pct_one_sided'
]]


,comparison,criterion,improvement_old_minus_new,improvement_percent,z_score,p_one_sided_improvement,significant_improvement_5pct_one_sided
0,IND -> BYM,WAIC,38834.741319,3.038833,3274.484104,0.000000e+00,True
1,IND -> BYM,DIC,39519.881687,3.095027,663.667024,0.000000e+00,True
2,IND -> BYM,AIC_proxy_at_posterior_mean,11148.360258,0.872583,176.192616,0.000000e+00,True
3,BYM -> BYM+lon,WAIC,1310.218027,0.105738,103.512092,0.000000e+00,True
4,BYM -> BYM+lon,DIC,1300.353962,0.105091,20.320949,4.196926e-92,True
5,BYM -> BYM+lon,AIC_proxy_at_posterior_mean,783.734374,0.061883,11.578171,2.658415e-31,True
6,IND -> BYM+lon,WAIC,40144.959345,3.141358,3431.549859,0.000000e+00,True
7,IND -> BYM+lon,DIC,40820.235648,3.196865,1609.932882,0.000000e+00,True
8,IND -> BYM+lon,AIC_proxy_at_posterior_mean,11932.094632,0.933926,492.374803,0.000000e+00,True


In [19]:
# Pooled-chain validation against Table 2 of the paper.
# This is intentionally computed from all 10 x 334 = 3,340 draws together;
# it is not the mean of the ten chain-specific WAIC/DIC values.
PAPER_TABLE2 = {
    'IND':     {'DIC': 1276928.89, 'WAIC': 1277919.03},
    'BYM':     {'DIC': 1204386.74, 'WAIC': 1239149.63},
    'BYM+lon': {'DIC': 1180995.01, 'WAIC': 1237861.46},
}
POOLED_CHECKPOINT = BASE_DIR / 'model_comparison_pooled_validation.json'
ABS_TOLERANCE = 0.05  # Paper values are printed to two decimals.

if POOLED_CHECKPOINT.exists():
    pooled_records = json.loads(POOLED_CHECKPOINT.read_text(encoding='utf-8'))
else:
    pooled_records = []
pooled_done = {r['model'] for r in pooled_records}

pooled_jobs = [
    ('IND', lambda: evaluate_ind(range(N_CHAINS))),
    ('BYM', lambda: evaluate_weekly(range(N_CHAINS), with_lon=False)),
    ('BYM+lon', lambda: evaluate_weekly(range(N_CHAINS), with_lon=True)),
]

for model, fn in pooled_jobs:
    if model in pooled_done:
        print(f'{model}: pooled validation already completed')
        continue
    metrics = fn()
    row = {'model': model, **{k: float(v) for k, v in metrics.items()}}
    pooled_records.append(row)
    POOLED_CHECKPOINT.write_text(json.dumps(pooled_records, indent=2), encoding='utf-8')
    print('completed pooled:', row)

pooled_results = pd.DataFrame(pooled_records)
validation_rows = []
for _, row in pooled_results.iterrows():
    model = row['model']
    for criterion in ['DIC', 'WAIC']:
        paper_value = PAPER_TABLE2[model][criterion]
        calculated = row[criterion]
        difference = calculated - paper_value
        validation_rows.append({
            'model': model,
            'criterion': criterion,
            'calculated': calculated,
            'paper_Table_2': paper_value,
            'difference': difference,
            'absolute_difference': abs(difference),
            'relative_difference_ppm': 1e6 * difference / paper_value,
            'matches_paper_within_tolerance': bool(abs(difference) <= ABS_TOLERANCE),
        })

pooled_validation = pd.DataFrame(validation_rows)
pooled_validation.to_csv(BASE_DIR / 'model_comparison_pooled_validation.csv', index=False)
pooled_validation


completed pooled: {'model': 'IND', 'WAIC': 1277919.038745681, 'lppd': -625770.470835756, 'p_WAIC': 13189.048537084464, 'DIC': 1276928.8955840885, 'D_bar': 1264311.1044854508, 'D_hat': 1251693.313386813, 'AIC_proxy_at_posterior_mean': 1277581.313386813, 'draws': 3340.0}


completed pooled: {'model': 'BYM', 'WAIC': 1239150.042920318, 'lppd': -605933.0615708929, 'p_WAIC': 13641.95988926605, 'DIC': 1204386.7592259094, 'D_bar': 1225200.9835131764, 'D_hat': 1246015.2078004435, 'AIC_proxy_at_posterior_mean': 1299455.2078004435, 'draws': 3340.0}


completed pooled: {'model': 'BYM+lon', 'WAIC': 1237861.46837546, 'lppd': -605512.3157568548, 'p_WAIC': 13418.418430875205, 'DIC': 1180995.010764524, 'D_bar': 1224148.9393453917, 'D_hat': 1267302.8679262593, 'AIC_proxy_at_posterior_mean': 1320762.8679262593, 'draws': 3340.0}


,model,criterion,calculated,paper_Table_2,difference,absolute_difference,relative_difference_ppm,matches_paper_within_tolerance
0,IND,DIC,1.276929e+06,1276928.89,0.005584,0.005584,0.004373,True
1,IND,WAIC,1.277919e+06,1277919.03,0.008746,0.008746,0.006844,True
2,BYM,DIC,1.204387e+06,1204386.74,0.019226,0.019226,0.015963,True
3,BYM,WAIC,1.239150e+06,1239149.63,0.412920,0.412920,0.333229,False
4,BYM+lon,DIC,1.180995e+06,1180995.01,0.000765,0.000765,0.000647,True
5,BYM+lon,WAIC,1.237861e+06,1237861.46,0.008375,0.008375,0.006766,True
